# 11 - Pilot: router + flag-gated text conditioning

3 conflict rows of one test photo (object, colour, material) x 4 arms, identical seed / 512 px / 20 steps. Nothing is trained.

| arm | what it is |
|---|---|
| `A_fixed` | `Pipeline.run(shaped=False)`, fixed guidance baseline |
| `B_router` | existing learned router (`models/router.pt`) |
| `D_flaggate` | A + flag-gated text (colour / material words blended toward the neutral caption by the governance flag) |
| `E_router_flaggate` | B + flag-gated text |

**Run the cells in order.** Set `SMOKE = False` in the last-but-one cell for the full pilot (the first pass with `SMOKE = True` is a 4-image smoke test; existing images are skipped on rerun).
Needs Drive: `pilot_v1.zip`, and `governance/governance.pt` (if missing, rule labels are used and the run says so).
Send back: the printed table, `contact_sheet.png`, `config.json`, `scores.csv`, `timings.json` (copied to Drive by the last cell).

In [ ]:
%%bash
pip uninstall -y torchao -q
pip install -q -U diffusers transformers accelerate safetensors peft lpips opencv-python-headless
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, sys, glob, json, zipfile, base64, zlib, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
GOV = f"{DRIVE_ROOT}/governance/governance.pt"
REPO = "/content/repo"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "governed-pipeline", "https://github.com/Harshitha-282007/IE643_final.git", REPO], check=True)
p = f"{REPO}/src/pipeline.py"
src = open(p).read()
if "text_flags" not in src:
    MIX = '''    @staticmethod
    def mix_text(pr, pooled, tn, pn, flags):
        """Flag-gated text conditioning: each attribute's neutral caption is blended in with weight = its flag."""
        for w, f in (("colour", float(flags[0])), ("material", float(flags[1]))):
            if tn.get(w) is not None and f:
                pr = pr - f * (pr - tn[w].to(pr.dtype))
                pooled = pooled - f * (pooled - pn[w].to(pooled.dtype))
        return pr, pooled

'''
    RUNBLK = '''        if text_flags is not None:
            nz = it.get("txt_neutral") or {}
            tn = {w: None if v is None else torch.cat([npe, v[0].to(dev)]) for w, v in nz.items()}
            pn = {w: None if v is None else torch.cat([nppe, v[1].to(dev)]) for w, v in nz.items()}
            pr, pooled = self.mix_text(pr, pooled, tn, pn, text_flags)
'''
    a = "    @torch.no_grad()\n    def run_role("
    b = "    def run(self, it, p=None, shaped=True, gates=None, split=True, style_only=None, guided=False, seed_off=0):"
    c = "        governed = gates is not None\n"
    assert src.count(a) == 1 and src.count(b) == 1 and src.count(c) == 1
    src = src.replace(a, MIX + a).replace(b, b[:-2] + ", text_flags=None):").replace(c, RUNBLK + c)
    open(p, "w").write(src)
open(f"{REPO}/src/pilot_flag_gate.py", "wb").write(zlib.decompress(base64.b64decode("eNqVWW2P2zYS/u5fQQgoVkoVZTe9uwbbEw7pJSmKS3PBtrgvxkKgLdpmLEs6kvaum9v/fs8MKVmyvdnESNcSOZwZzsszM24URZOPumrctSgbZcWiksvnS+lUKZy6d2Le1KV2uql1vRSbZqewUjVbI16IDaiMllUqZF3605oObNrGKnGn3Uq4lRLqXltHp02zxYF/TCb/rhWYWyfaVeOaFFRGEd96Uem5A92dFXEz+6TmLg3i0l5akooFyZdmY1OhS1U7PZeVsAoqvxBG2abaksJ4sU619noyEfi8Lhb6HiTDz0fdqkrXKjPbOrYr2aoyfycrqxLx5Y9ntdzqUtZzJWbSMh8R//Pthz/e3iQs8efC3/hRiW0eCFppJN0mqPCH2Z5o0FuxUtLUEP7aOaNnOC5uwIR23su9Miz5TUFuJC9+xV1TdjSfsHm8KGBwmLiAvRNW4rX4/iQqYguBFXy2kvVSiZ0Vr/2d34Y7DxX46jsPNSHRP5+THI5TyDF/1ywVwswkk8k7OkgBqMTCNBuOviVC1tTspVq5u8asRbxpSlVZgRB5Y/ROJT8JvaDQ1VZsa7mTupIzXI6Omy0e8Er00iixtVCEZPvNGmcmvF1h/aIxcg56vsBFJj40bkWOAV9nJCxQZuLXjVwqXLtV5rmmZ+H0BkRY2ih4dG579pQQesli74x2TtVQr8TdYYGi1OanicLd9iKwWUnno8SSQLvWLewK+zZCIitIV+TGdqMOAqxCuspazChJwb6EgpMIgKCRwsaJT7apu+fGdk/QV026l3q7afdCWlG33VIL9ljAv7acsB8+/vpehE2+fn9a7WSFgKkUk7+98eQDlwW6zgFWvD8myRZKuq0h7PG0EnlRtCuDlPS0xJ8iJXPNWtU9Xa22cEpVzGVLaDGZvL757XeRi2kUkCJKRdSlMD0fkoreTgI9up3869cPb4hFHAG8iAipRF9IpQjhOSnVQrR6vi4Y92JCjYLADnFPquk/FXDOtpV2eUT4iLOEavlVcs3JBde80wa4eQBPZCIug9jTZeIhd4yb7OsOOsVO4qtGgNytCKKPgxsJQeS8yQJPoF4ggUrbBU0wobYU2O8YEmdq0YCTrLuwRCguVa0MZXBGsUV8GeFhagMoN8IIXYv3mbqnyGGDDE2DRCncvlX5VbAG/UnFWtelzdniCaWvmUZsuehW5Lk34i3Lmu0h6fMDP/fSiLW3qSfJkArwjtxWLganxuilrhEcuoxuU5xOpm9vMhJZNIvYJJAhTM+x0SXx9I6IZ/vkwHkJwtl+Corbfg3K1o2j5IuD+n/P+W05OOitj9JWb9XxyXg55QC7nUYUeJYfCl7xsDjlwBtv80q3TQE53uaVr5E/zpp4ELgHsWGTTBf5EIoSCoQPDUokDPZlJkG5IZMu/A5sntDUKGBCTZ5JxXQ5Xd+yo9bkJra59wYwGVlws60J0t4a05g4qpthdlnptF1ooIvHywqpRa0FZU6f0oDWoqUWKv6YCh+0pXSSIDrtsDql6pOT5mnog8LLWnfL1NDkBEKpqJpl3hpdu+CRR2CR6xih4i+/HcjaUG07IkqXArdnisZmG7lWUMjGvWZcNIpm7fsOT9cicCu5mZWoHVDy2gN31rSqjhfR5+5+Dy8+G1j3IfLHPmatUSjsqvAoVLCgqclQouPIqEVYhx/hjii8jNPy1rOyrZozRpRoCeO12ufIS5+Odq3cfJU3bQzVIv8WoS0E/24Rj8oomIrWQyQRg0FQQXgeFAuriEenlo3ZA3eZNrzdJunktAXssYiMewIaXp1w3fyMAVIuVPmgWgFXkiNTsFjt1IYswYb4U7ewp7fEEXEK85NzCz4QswGTJPHdr64XzZM4iBYo9+I6EQfU4loBBgiFRdVIF++8sutU7JhTDyeZl5889GcRFCXO/vIbhQddo2DSGCmBDt5No3sYzKMLN2EEcowVKEyKn3pW6L/AiRmC3n8fHyBVD3fCxfvbdFYkJM9HiP5YlByHxteHTAjTwcnh/rmIGnwoUgg5jR3Gjl8JoV50oDikOABlKJZkXXqgu5BdQptPjymbLyzQ4xM6MWFBDXAeBT+C7+NuiHw3HA+6jCQaDRtTMOtrUyrCG+P/kwbyGOq/Mj9LxNr5VsBPCGcDKenZAmPjHlKo081K9LI2pggfBc2oesQRxQz3diFICG16w9Lbwar0drguU/puMnno8pLqB/qjGOBPqYD/BkXYLeA0QmI9HfKBvlgIrEZ9BbhQ99N3sOMqGYriR57DSNZo4j3Hp29/n2CEoa4dT6/nuA0a6G9QbBQxbnGW82kz/q0KPyrF9wj/kdU2dAiQ2WFqmIcaM1/xgmslGvBcoECG4vrwIkx3GUWYb36p2SA855gDlJYxV1U+nHAAo0rTS+anubDjY/jLAE7LZBNsUDcxNsIZ5Wj+evEZpFzHL3R5cfuQtfUyGh081YgVuj7J0JM27KTlCGzwUsuN8nzONiB9CpBxs/m2lJnd1/OVaahRjDGyu0tchmyZ0Z94fGyQUsNqlo4zO8ms3AUtvlYqOW8KG5LR/tcbLeKJoMF4Hg9UEs+hZSpejpn3WMOkaBYP7gdC3BF0A2JU7TD3jQ4SYkXfPb96ZcV3z1++xN/samEj8Z3wd+3R7FElkySMdfzDQiiGmJ1zZAZpwr+X4YW/Q0eK8c4Rf5v45jZ/tP0g2+b05xx2zxfAeVwKoM1S5za/ymCcVVOV+WX2I463+P5rSpT5j9klV7lK3mPxB6/2wXD+AsFyw4D2Gz7Zjo2ZijDh5daZkMImNJmh01pQOz4sHOPUOpNxT7VNPF5KblrqNpNWGiP38biZHqdjwO6jhEwyXA39P+r8zS8/07XADgPC1d/GMfJFBPCgBVW+JP8cHBxLP+Fq5+AK22V2jvE/1hSNvrUL7fmtf0OPQo/jSnxA6bnNZAutyphjRJeHpv9cywZNc478Vt/jBotFsbPF69z3p2TumY0HVieteqshN8kvSbZRso6T5Il+gz7PnlFvsBu1vXbetbvddK6trq2jMS3eUUYZLvYdD/p5qC2zN5if3hmCQLr0YDtzTTG3u5Fb2KY2w3Lko/n+uGSH+saa+DxMxe+Tx3a4ofajKyG2RD9uVwotZKDyI2w/HvZzaUD8bviUrsXMW+nZ+dWs3dMT/w5YOV+1KGflPZmgQqhsZ0Rg4wphSDLh0Zfie1F57KZXHLAA3jx+mb0Sz0Q82sb+D1jsT1PL+9+tUqAfmIecpVOfuqrebviXKH/g6wYfeT/F+UuMNRu7au7iYVCjIPTbVrnCaYeGF74bBerDtaB0Cg3jxe30+i+XQGP6/xi14/sNktjzuxqJ46zpZV2NZWGT+PMMcHGYEi6Sh3S03I0G2HhENJnqU9rBx8FYbO0xknhNyB2fek2/GVSS4y5DUzN3eYpZI2mHu3Pq9xd5dfC3ZPi7zwyqexUPVJd8+t7p+drGUzYpr+z7FSal6HR6uXJFJfe4CBV+jljwQ0geV5xDBvG9UGdanV9dXSbDDPzymcn/ASMfnuQ=")))
sys.path.insert(0, f"{REPO}/src")
LOCAL = "/content/pilot_v1"
if not glob.glob(f"{LOCAL}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL, exist_ok=True)
    zipfile.ZipFile(f"{DRIVE_ROOT}/pilot_v1.zip").extractall(LOCAL)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
print("patched:", "text_flags" in open(p).read(), "| base rows:", len(BASE_ROWS), "| governance.pt:", os.path.exists(GOV))

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from transformers import CLIPTokenizer
from pipeline import Pipeline, BASE_ID
import pilot_flag_gate as PF
from governance import model as GM
from governance.evalkit import EvalKit
from router import RouterEnsemble
tok = CLIPTokenizer.from_pretrained(BASE_ID, subfolder="tokenizer")
OID, ROWS = PF.pick_photo(BASE_ROWS, tok)
print("photo:", OID, [(PF.ER.kind_of(r), r["caption"]) for r in ROWS])
P = Pipeline(res=512, steps=20)
router = RouterEnsemble(f"{REPO}/models/router.pt")
net = GM.load(GOV) if os.path.exists(GOV) else None
KIT = EvalKit(rule="reference")
print("ready | governance net:", net is not None, "| GPU GB", round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
SMOKE = True            # True: colour row only (4 images). False: all 3 rows (resumable, existing images are skipped)
OUT = "/content/pilot_out"
rows = [r for r in ROWS if PF.ER.kind_of(r) == "col"] if SMOKE else ROWS
items, info, S = PF.run_pilot(P, rows, DATA_DIR, OUT, net=net, router=router, kit=KIT)
pd.set_option("display.width", 250)
cols = ["id", "kind", "arm", "pix_diff_vs_A", "edge_iou", "col_hit", "col_share", "col_leak", "mat_hit", "obj_hit", "sem_margin", "id_leak", "dino_ref", "clip_t"]
print(S[[c for c in cols if c in S]].round(3).to_string(index=False))
print(PF.contact_sheet(items, rows, OUT))

In [ ]:
import shutil
DEST = f"{DRIVE_ROOT}/sdxl_runs/pilot_flaggate_v1"
shutil.copytree(OUT, DEST, dirs_exist_ok=True)
print("saved to", DEST, sorted(os.listdir(DEST)))